# Native benchmarks before and after WarBias unlearning

The default Ukrainian tasks evaluate five unchanged bases and ten Ukrainian-trained BiasUnlearn adapters across both groups. Evaluate each adapter only on tasks in its training language. Each baseline is shared across both group directions for that model.

Fair-UK collects predictions and calculates every metric, change and uncertainty interval. See the [evaluation setup](NATIVE_EVALUATION.md).

In [1]:
import json
import os
import sys
from pathlib import Path

repo = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs/unlearning/models.json").is_file()
)
sys.path.insert(0, str(repo / "src"))
from fairforget.unlearning import native_evaluation  # noqa: E402

settings_path = repo / "configs/unlearning/local.json"
settings = json.loads(settings_path.read_text()) if settings_path.exists() else {}

## Evaluation inputs

Select a current Fair-UK evaluation bundle and a separate development bundle. The default native benchmarks are StereoSet UK, BBQ UK, both WinoBias UK views and WinoPron UK. Select other approved native views by their task IDs. English-trained adapters require English task inputs and an English qualification task in a separate run.

The final annotated WarBias original/neutral panels use the Fair-UK WarBias adapter. Keep those results separate from this native benchmark table. All benchmark inputs here are evaluation only.

In [2]:
FAIR_UK_ROOT = Path(os.environ.get("FAIR_UK_ROOT", repo.parent / "fair-uk"))
FAIR_UK_PYTHON = Path(os.environ.get("FAIR_UK_PYTHON", repo / ".venv-evaluation/bin/python"))
EVALUATION_BUNDLE = Path(
    os.environ.get("FAIR_UK_EVALUATION_BUNDLE", repo / "data/evaluation/manifest.json")
)
QUALIFICATION_BUNDLE = Path(
    os.environ.get("FAIR_UK_QUALIFICATION_BUNDLE", repo / "data/development/manifest.json")
)
EVALUATION_ROOT = Path(os.environ.get("FAIR_UK_OUTPUT", repo / "outputs/native_evaluation"))
MODELS_ROOT = Path(
    os.environ.get("FAIRFORGET_MODELS_ROOT", settings.get("models_root", repo / "models"))
)
TRAINING_ROOT = Path(
    os.environ.get(
        "FAIRFORGET_TRAINING_ROOT",
        settings.get("output_root", repo / "outputs/unlearning"),
    )
)
TASKS = native_evaluation.TASKS
QUALIFICATION_TASK = "stereoset_uk"
RUN_BEFORE = False
RUN_AFTER = False
RUN_COMPARISON = False

os.environ["VLLM_BATCH_INVARIANT"] = "1"

workflow = native_evaluation.NativeEvaluation(
    repo,
    representation="adapter",
    attention_backend="TRITON_ATTN",
    fair_uk_root=FAIR_UK_ROOT,
    python=FAIR_UK_PYTHON,
    bundle=EVALUATION_BUNDLE,
    development=QUALIFICATION_BUNDLE,
    output=EVALUATION_ROOT,
    models_root=MODELS_ROOT,
    training_root=TRAINING_ROOT,
    tasks=TASKS,
    qualification_task=QUALIFICATION_TASK,
    bootstrap=1000,
    seed=42,
    tolerance=1e-6,
)

## Run plan and validation

The [study configuration](../configs/unlearning/warbias_final.json) selects the methods and directions. Training directories use `METHOD/MODEL__GROUP_LANGUAGE/`. Each post checkpoint must have the correct base, group, language, recipe, export checks and file hashes.

Collection verifies active source files and exact prediction coverage. Development and evaluation IDs must be separate. Existing predictions and logs are never overwritten.

In [3]:
assert len(workflow.models) == 5
assert len(workflow.plan) == 20
assert {item["method"] for item in workflow.plan} == {"biasunlearn"}
assert len(workflow.active_plan) == 10
for item in workflow.active_plan:
    print(item["method"], item["run_id"], workflow.tasks_for(item))
print("Shared base checkpoints:", len(workflow.models))
print("Post-training checkpoints for these tasks:", len(workflow.active_plan))
print("Native tasks:", ", ".join(workflow.tasks))

biasunlearn qwen3_8b__idp_uk ('stereoset_uk', 'bbq_uk', 'winobias_uk_natural', 'winobias_uk_controlled', 'winopron_uk')
biasunlearn qwen3_8b__veteran_uk ('stereoset_uk', 'bbq_uk', 'winobias_uk_natural', 'winobias_uk_controlled', 'winopron_uk')
biasunlearn llama31_8b_instruct__idp_uk ('stereoset_uk', 'bbq_uk', 'winobias_uk_natural', 'winobias_uk_controlled', 'winopron_uk')
biasunlearn llama31_8b_instruct__veteran_uk ('stereoset_uk', 'bbq_uk', 'winobias_uk_natural', 'winobias_uk_controlled', 'winopron_uk')
biasunlearn gemma3_12b_it__idp_uk ('stereoset_uk', 'bbq_uk', 'winobias_uk_natural', 'winobias_uk_controlled', 'winopron_uk')
biasunlearn gemma3_12b_it__veteran_uk ('stereoset_uk', 'bbq_uk', 'winobias_uk_natural', 'winobias_uk_controlled', 'winopron_uk')
biasunlearn lapa12b_instruct__idp_uk ('stereoset_uk', 'bbq_uk', 'winobias_uk_natural', 'winobias_uk_controlled', 'winopron_uk')
biasunlearn lapa12b_instruct__veteran_uk ('stereoset_uk', 'bbq_uk', 'winobias_uk_natural', 'winobias_uk_cont

## Fair-UK implementation

The [collector](../src/fairforget/unlearning/native_evaluation.py) runs the Fair-UK `run` command in the evaluation environment. Each checkpoint runs in a separate process. `paper compare-native` verifies matched sources, effective tokenizer inputs and inference settings before calculating paired native metrics.

Before collection, each unchanged checkpoint runs twice on development data. Candidate likelihoods must be finite and agree within an absolute tolerance of 1e-6. This is an engineering check, separate from scientific thresholds.

In [4]:
print("Collection backend: Fair-UK / vLLM")
print("Model settings:", native_evaluation.MODEL_SETTINGS)
print("Evaluation seed:", workflow.seed)
print("Metric and interval implementation: Fair-UK")

Collection backend: Fair-UK / vLLM
Model settings: dtype=bfloat16,tensor_parallel_size=1,max_model_len=4096,enable_thinking=False,enforce_eager=True
Evaluation seed: 42
Metric and interval implementation: Fair-UK


## 1. Evaluate before unlearning

Enable `RUN_BEFORE` after configuring input bundles, the evaluation environment and GPU allocation. Verify all five pinned base checkpoints, run the repeatability checks, then collect each selected task once per model.

In [5]:
if RUN_BEFORE:
    workflow.run_before()
else:
    print("Before collection is disabled. Five shared baselines are configured.")

Before collection is disabled. Five shared baselines are configured.


## 2. Evaluate after unlearning

Run the [BiasUnlearn notebook](biasunlearn.ipynb), then enable `RUN_AFTER`. Verify the ten Ukrainian exports and their training receipts before collecting the default tasks. Each adapter uses only tasks in its training language.

In [6]:
if RUN_AFTER:
    workflow.run_after()
else:
    print("After collection is disabled. Matching checkpoints:", len(workflow.active_plan))

After collection is disabled. Matching checkpoints: 10


## 3. Compare before and after

Enable `RUN_COMPARISON` after collecting both phases. Fair-UK compares each intervention with its model's shared base. It preserves task denominators, undefined values, component dependence and uncertainty status. Comparisons run on saved predictions without GPUs.

In [7]:
if RUN_COMPARISON:
    workflow.run_comparison()
else:
    print("Saved-run comparison is disabled.")

Saved-run comparison is disabled.


## Results

Read the Fair-UK paired reports. Each row identifies the method, model, target group, training language, evaluation task, native metric, unit, change and uncertainty. Export the complete table to a new CSV path.

In [8]:
summary = workflow.summary()
if summary:
    csv_path = workflow.output / "native_before_after.csv"
    native_evaluation.export_summary(summary, csv_path)
    print("Native result rows:", len(summary))
    print("Saved:", csv_path.name)
else:
    print("No paired reports are present in this evaluation directory.")

No paired reports are present in this evaluation directory.
